# 🚴 Titan Training - Detección y Clasificación de Ciclistas

Este notebook entrena dos modelos:
1. **Modelo de Detección (YOLOv11)**: Detecta objetos (bicicleta, casco, ropa, números)
2. **Modelo de Clasificación Multi-label**: Clasifica atributos (colores, marcas, números)

## 📋 Requisitos
- Datasets en Google Drive:
  - `MyDrive/titan_project/models_datasets/titan_detection.yolov11/`
  - `MyDrive/titan_project/models_datasets/titan_labels.multiclass/`

---

## 1️⃣ Configuración Inicial

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Verificar GPU
import torch
print(f"🖥️  PyTorch version: {torch.__version__}")
print(f"🎮 CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Instalar dependencias
!pip install -q ultralytics>=8.3.0
!pip install -q torchvision>=0.15.0
!pip install -q pandas seaborn tqdm pyyaml

In [ ]:
# Configurar rutas
import os
from pathlib import Path

# Rutas base
DRIVE_PATH = Path('/content/drive/MyDrive/titan_project')
DATASETS_PATH = DRIVE_PATH / 'models_datasets'

# Rutas de datasets
DETECTION_DATASET = DATASETS_PATH / 'titan_detection.yolov11'
CLASSIFICATION_DATASET = DATASETS_PATH / 'titan_labels.multiclass'

# Verificar que existen
print("📁 Verificando datasets...")
print(f"   Detección: {DETECTION_DATASET.exists()} - {DETECTION_DATASET}")
print(f"   Clasificación: {CLASSIFICATION_DATASET.exists()} - {CLASSIFICATION_DATASET}")

# Listar contenido
if DETECTION_DATASET.exists():
    print(f"\n📂 Contenido de detección:")
    for item in DETECTION_DATASET.iterdir():
        print(f"   - {item.name}")

if CLASSIFICATION_DATASET.exists():
    print(f"\n📂 Contenido de clasificación:")
    for item in CLASSIFICATION_DATASET.iterdir():
        print(f"   - {item.name}")

## 2️⃣ Análisis de Datasets

In [ ]:
# Analizar dataset de detección
import yaml

print("="*60)
print("📊 ANÁLISIS DATASET DE DETECCIÓN")
print("="*60)

# Leer data.yaml
data_yaml_path = DETECTION_DATASET / 'data.yaml'
if data_yaml_path.exists():
    with open(data_yaml_path) as f:
        data_config = yaml.safe_load(f)
    
    print(f"\n🏷️  Clases ({data_config['nc']}):")
    for i, name in enumerate(data_config['names']):
        print(f"   {i}: {name}")

# Contar imágenes por split
print(f"\n📁 Imágenes por split:")
for split in ['train', 'valid', 'test']:
    split_path = DETECTION_DATASET / split / 'images'
    if split_path.exists():
        n_images = len(list(split_path.glob('*')))
        print(f"   {split}: {n_images} imágenes")

In [ ]:
# Analizar dataset de clasificación
import pandas as pd

print("="*60)
print("📊 ANÁLISIS DATASET DE CLASIFICACIÓN")
print("="*60)

# Buscar archivo de clases
classes_file = None
for name in ['_classes.csv', 'classes.csv']:
    path = CLASSIFICATION_DATASET / name
    if path.exists():
        classes_file = path
        break
    # Buscar en subdirectorios
    for subdir in ['train', 'valid', 'test']:
        path = CLASSIFICATION_DATASET / subdir / name
        if path.exists():
            classes_file = path
            break

if classes_file:
    df = pd.read_csv(classes_file)
    print(f"\n📄 Archivo de clases: {classes_file.name}")
    print(f"   Filas (imágenes): {len(df)}")
    print(f"   Columnas (clases + filename): {len(df.columns)}")
    
    # Clases (excluyendo filename)
    class_cols = [c for c in df.columns if c != 'filename']
    print(f"   Número de atributos: {len(class_cols)}")
    
    # Estadísticas
    print(f"\n📈 Estadísticas:")
    class_counts = df[class_cols].sum().sort_values(ascending=False)
    print(f"   Top 10 atributos más frecuentes:")
    for name, count in class_counts.head(10).items():
        print(f"      {name}: {int(count)}")
    
    print(f"\n   Atributos con 0 muestras: {(class_counts == 0).sum()}")
    print(f"   Promedio de atributos por imagen: {df[class_cols].sum(axis=1).mean():.2f}")
else:
    print("⚠️  No se encontró archivo de clases")

# Contar imágenes por split
print(f"\n📁 Imágenes por split:")
for split in ['train', 'valid', 'test']:
    split_path = CLASSIFICATION_DATASET / split
    if split_path.exists():
        n_images = len(list(split_path.glob('*.jpg'))) + len(list(split_path.glob('*.png')))
        print(f"   {split}: {n_images} imágenes")

In [ ]:
# Visualizar algunas imágenes del dataset
import matplotlib.pyplot as plt
from PIL import Image
import random

def show_sample_images(dataset_path, split='train', n=4):
    """Muestra imágenes de ejemplo del dataset."""
    img_dir = dataset_path / split
    if not img_dir.exists():
        img_dir = dataset_path / split / 'images'
    
    if not img_dir.exists():
        print(f"No se encontró directorio de imágenes en {dataset_path / split}")
        return
    
    images = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    if len(images) == 0:
        print("No se encontraron imágenes")
        return
    
    samples = random.sample(images, min(n, len(images)))
    
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    if n == 1:
        axes = [axes]
    
    for ax, img_path in zip(axes, samples):
        img = Image.open(img_path)
        ax.imshow(img)
        ax.set_title(img_path.name[:20] + '...' if len(img_path.name) > 20 else img_path.name)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

print("📸 Muestras del dataset de detección:")
show_sample_images(DETECTION_DATASET, 'train', 4)

print("\n📸 Muestras del dataset de clasificación:")
show_sample_images(CLASSIFICATION_DATASET, 'train', 4)

---
## 3️⃣ Entrenamiento del Modelo de Detección (YOLOv11)

In [ ]:
from ultralytics import YOLO

# Configuración de entrenamiento
DETECTION_CONFIG = {
    'model': 'yolo11m.pt',  # Modelo medio para balance precisión/velocidad
    'epochs': 100,
    'batch': 16,  # Ajustar según GPU (T4: 16, A100: 32)
    'imgsz': 640,
    'patience': 15,
    'workers': 2,
    'optimizer': 'AdamW',
    'lr0': 0.001,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3.0,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'degrees': 0.0,
    'translate': 0.1,
    'scale': 0.5,
    'flipud': 0.0,
    'fliplr': 0.5,
    'mosaic': 1.0,
    'mixup': 0.1,
    'save_period': 10,
    'project': str(DRIVE_PATH / 'runs' / 'detection'),
    'name': 'titan_detection_v1',
    'exist_ok': True,
    'pretrained': True,
    'verbose': True
}

print("📋 Configuración de entrenamiento de detección:")
for k, v in list(DETECTION_CONFIG.items())[:10]:
    print(f"   {k}: {v}")
print("   ...")

In [ ]:
# Entrenar modelo de detección
print("\n" + "="*60)
print("🚀 INICIANDO ENTRENAMIENTO DE DETECCIÓN (YOLOv11)")
print("="*60)

# Cargar modelo
detection_model = YOLO(DETECTION_CONFIG['model'])

# Entrenar
detection_results = detection_model.train(
    data=str(DETECTION_DATASET / 'data.yaml'),
    **{k: v for k, v in DETECTION_CONFIG.items() if k != 'model'}
)

print("\n" + "="*60)
print("✅ ENTRENAMIENTO DE DETECCIÓN COMPLETADO")
print("="*60)

In [ ]:
# Validar modelo de detección
print("\n🔍 Validando modelo de detección en test set...")

# Cargar mejor modelo
best_detection_path = Path(DETECTION_CONFIG['project']) / DETECTION_CONFIG['name'] / 'weights' / 'best.pt'
if best_detection_path.exists():
    best_detection = YOLO(str(best_detection_path))
    
    # Validar en test
    test_results = best_detection.val(
        data=str(DETECTION_DATASET / 'data.yaml'),
        split='test',
        plots=True
    )
    
    print(f"\n📊 Resultados en Test Set:")
    print(f"   mAP@0.5:      {test_results.box.map50:.4f}")
    print(f"   mAP@0.5:0.95: {test_results.box.map:.4f}")
    print(f"   Precision:    {test_results.box.mp:.4f}")
    print(f"   Recall:       {test_results.box.mr:.4f}")
else:
    print(f"⚠️  Modelo no encontrado en: {best_detection_path}")

---
## 4️⃣ Entrenamiento del Modelo de Clasificación Multi-label

In [ ]:
# Código del modelo de clasificación (inline para Colab)
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss


class MultiLabelDataset(Dataset):
    """Dataset para clasificación multi-label."""
    
    def __init__(self, data_dir, split='train', transform=None, label_file='_classes.csv'):
        self.data_dir = Path(data_dir)
        self.split = split
        self.transform = transform
        
        # Cargar labels
        self.labels_df = self._load_labels(label_file)
        self.samples = self._get_samples()
        self.class_names = [col for col in self.labels_df.columns if col != 'filename']
        self.num_classes = len(self.class_names)
        
        print(f"📁 Dataset {split}: {len(self.samples)} imágenes, {self.num_classes} clases")
    
    def _load_labels(self, label_file):
        for path in [self.data_dir / label_file, self.data_dir / 'train' / label_file]:
            if path.exists():
                return pd.read_csv(path)
        raise FileNotFoundError(f"No se encontró {label_file}")
    
    def _get_samples(self):
        samples = []
        split_dir = self.data_dir / self.split
        
        for img_path in split_dir.iterdir():
            if img_path.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
                row = self.labels_df[self.labels_df['filename'] == img_path.name]
                if len(row) == 0:
                    row = self.labels_df[self.labels_df['filename'].str.startswith(img_path.stem)]
                
                if len(row) > 0:
                    label_cols = [c for c in self.labels_df.columns if c != 'filename']
                    labels = row[label_cols].values[0].astype(np.float32)
                    samples.append((img_path, labels))
        
        return samples
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, labels = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(labels, dtype=torch.float32)
    
    def get_class_weights(self):
        all_labels = np.array([s[1] for s in self.samples])
        pos_counts = all_labels.sum(axis=0)
        neg_counts = len(all_labels) - pos_counts
        pos_weights = np.where(pos_counts > 0, neg_counts / pos_counts, 1.0)
        return torch.tensor(np.clip(pos_weights, 0.1, 10.0), dtype=torch.float32)


class MultiLabelClassifier(nn.Module):
    """Clasificador multi-label con backbone preentrenado."""
    
    def __init__(self, num_classes, backbone='efficientnet_b2', pretrained=True, dropout=0.3):
        super().__init__()
        
        if backbone == 'efficientnet_b2':
            self.backbone = models.efficientnet_b2(weights='DEFAULT' if pretrained else None)
            num_features = 1408
            self.backbone.classifier = nn.Identity()
        elif backbone == 'resnet50':
            self.backbone = models.resnet50(weights='DEFAULT' if pretrained else None)
            num_features = 2048
            self.backbone.fc = nn.Identity()
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(num_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout / 2),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        if features.dim() > 2:
            features = features.mean(dim=[2, 3])
        return self.classifier(features)


print("✅ Clases definidas")

In [ ]:
# Configuración de clasificación
CLASSIFICATION_CONFIG = {
    'backbone': 'efficientnet_b2',
    'epochs': 80,
    'batch_size': 32,
    'imgsz': 224,
    'patience': 12,
    'learning_rate': 0.0003,
    'weight_decay': 0.01,
    'dropout': 0.3,
    'save_dir': str(DRIVE_PATH / 'runs' / 'classification' / 'titan_classification_v1')
}

print("📋 Configuración de entrenamiento de clasificación:")
for k, v in CLASSIFICATION_CONFIG.items():
    print(f"   {k}: {v}")

In [ ]:
# Preparar datasets
def get_transforms(split, imgsz=224):
    if split == 'train':
        return T.Compose([
            T.Resize((imgsz + 32, imgsz + 32)),
            T.RandomCrop(imgsz),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(15),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    else:
        return T.Compose([
            T.Resize((imgsz, imgsz)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

# Crear datasets
train_dataset = MultiLabelDataset(
    CLASSIFICATION_DATASET, split='train',
    transform=get_transforms('train', CLASSIFICATION_CONFIG['imgsz'])
)

val_dataset = MultiLabelDataset(
    CLASSIFICATION_DATASET, split='valid',
    transform=get_transforms('valid', CLASSIFICATION_CONFIG['imgsz'])
)

# Test dataset (si existe)
test_dataset = None
if (CLASSIFICATION_DATASET / 'test').exists():
    test_dataset = MultiLabelDataset(
        CLASSIFICATION_DATASET, split='test',
        transform=get_transforms('test', CLASSIFICATION_CONFIG['imgsz'])
    )

# Crear DataLoaders
train_loader = DataLoader(train_dataset, batch_size=CLASSIFICATION_CONFIG['batch_size'],
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CLASSIFICATION_CONFIG['batch_size'],
                        shuffle=False, num_workers=2, pin_memory=True)

print(f"\n✅ DataLoaders creados")
print(f"   Train: {len(train_loader)} batches")
print(f"   Valid: {len(val_loader)} batches")

In [ ]:
# Entrenar modelo de clasificación
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Dispositivo: {device}")

# Crear modelo
classification_model = MultiLabelClassifier(
    num_classes=train_dataset.num_classes,
    backbone=CLASSIFICATION_CONFIG['backbone'],
    dropout=CLASSIFICATION_CONFIG['dropout']
).to(device)

print(f"\n🔧 Modelo creado:")
print(f"   Clases: {train_dataset.num_classes}")
print(f"   Parámetros: {sum(p.numel() for p in classification_model.parameters()):,}")

# Loss con pesos
pos_weight = train_dataset.get_class_weights().to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Optimizador y scheduler
optimizer = AdamW(classification_model.parameters(), 
                  lr=CLASSIFICATION_CONFIG['learning_rate'],
                  weight_decay=CLASSIFICATION_CONFIG['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=CLASSIFICATION_CONFIG['epochs'])

# Crear directorio de guardado
save_dir = Path(CLASSIFICATION_CONFIG['save_dir'])
save_dir.mkdir(parents=True, exist_ok=True)

print("✅ Optimizador y scheduler configurados")

In [ ]:
# Loop de entrenamiento
print("\n" + "="*60)
print("🚀 INICIANDO ENTRENAMIENTO DE CLASIFICACIÓN")
print("="*60)

history = {'train_loss': [], 'val_loss': [], 'val_f1': []}
best_f1 = 0.0
patience_counter = 0

for epoch in range(CLASSIFICATION_CONFIG['epochs']):
    # Training
    classification_model.train()
    train_loss = 0.0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{CLASSIFICATION_CONFIG["epochs"]} [Train]')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = classification_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(classification_model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    train_loss /= len(train_loader)
    
    # Validation
    classification_model.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f'Epoch {epoch+1} [Valid]'):
            images, labels = images.to(device), labels.to(device)
            outputs = classification_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            probs = torch.sigmoid(outputs)
            all_preds.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    
    val_loss /= len(val_loader)
    
    # Calcular métricas
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    preds_binary = (all_preds >= 0.5).astype(int)
    
    f1_macro = f1_score(all_labels, preds_binary, average='macro', zero_division=0)
    f1_micro = f1_score(all_labels, preds_binary, average='micro', zero_division=0)
    h_loss = hamming_loss(all_labels, preds_binary)
    
    # Guardar historial
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(f1_macro)
    
    # Imprimir métricas
    print(f"\nEpoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")
    print(f"          F1 Macro={f1_macro:.4f}, F1 Micro={f1_micro:.4f}, Hamming={h_loss:.4f}")
    
    # Guardar mejor modelo
    if f1_macro > best_f1:
        best_f1 = f1_macro
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': classification_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_f1': best_f1,
            'class_names': train_dataset.class_names,
            'config': CLASSIFICATION_CONFIG
        }, save_dir / 'best.pt')
        print(f"          ✅ Mejor modelo guardado! (F1: {best_f1:.4f})")
    else:
        patience_counter += 1
    
    # Early stopping
    if patience_counter >= CLASSIFICATION_CONFIG['patience']:
        print(f"\n⏹️  Early stopping en época {epoch+1}")
        break
    
    scheduler.step()

# Guardar modelo final
torch.save({
    'epoch': epoch,
    'model_state_dict': classification_model.state_dict(),
    'class_names': train_dataset.class_names,
    'config': CLASSIFICATION_CONFIG
}, save_dir / 'last.pt')

print("\n" + "="*60)
print("✅ ENTRENAMIENTO DE CLASIFICACIÓN COMPLETADO")
print("="*60)
print(f"   Mejor F1 Macro: {best_f1:.4f}")
print(f"   Modelo guardado en: {save_dir}")

In [ ]:
# Graficar historial de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['val_f1'], label='F1 Macro', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('Validation F1 Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(save_dir / 'training_history.png', dpi=150)
plt.show()

---
## 5️⃣ Prueba de Inferencia Combinada

In [ ]:
# Función de inferencia combinada
def combined_inference(image_path, detection_model, classification_model, class_names, device):
    """Ejecuta detección y clasificación en una imagen."""
    from PIL import Image
    import torchvision.transforms as T
    
    # Detección
    det_results = detection_model.predict(image_path, conf=0.25, verbose=False)
    
    # Parsear detecciones
    detections = []
    if det_results[0].boxes is not None:
        for box in det_results[0].boxes:
            detections.append({
                'class': detection_model.names[int(box.cls)],
                'confidence': float(box.conf),
                'bbox': box.xyxy[0].cpu().numpy().tolist()
            })
    
    # Clasificación
    img = Image.open(image_path).convert('RGB')
    transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    img_tensor = transform(img).unsqueeze(0).to(device)
    
    classification_model.eval()
    with torch.no_grad():
        logits = classification_model(img_tensor)
        probs = torch.sigmoid(logits)[0].cpu().numpy()
    
    # Parsear clasificaciones
    classifications = []
    for i, (name, prob) in enumerate(zip(class_names, probs)):
        if prob >= 0.5:
            classifications.append({'attribute': name, 'probability': float(prob)})
    
    classifications.sort(key=lambda x: x['probability'], reverse=True)
    
    return {
        'detections': detections,
        'classifications': classifications
    }

print("✅ Función de inferencia combinada definida")

In [ ]:
# Cargar mejores modelos
print("📂 Cargando mejores modelos...")

# Detección
best_det_path = Path(DETECTION_CONFIG['project']) / DETECTION_CONFIG['name'] / 'weights' / 'best.pt'
if best_det_path.exists():
    best_detection = YOLO(str(best_det_path))
    print(f"   ✅ Detección cargada: {best_det_path}")
else:
    print(f"   ⚠️  Modelo de detección no encontrado")
    best_detection = None

# Clasificación
best_cls_path = save_dir / 'best.pt'
if best_cls_path.exists():
    checkpoint = torch.load(best_cls_path)
    best_classification = MultiLabelClassifier(
        num_classes=len(checkpoint['class_names']),
        backbone=checkpoint['config']['backbone']
    ).to(device)
    best_classification.load_state_dict(checkpoint['model_state_dict'])
    best_classification.eval()
    class_names = checkpoint['class_names']
    print(f"   ✅ Clasificación cargada: {best_cls_path}")
else:
    print(f"   ⚠️  Modelo de clasificación no encontrado")
    best_classification = None

In [ ]:
# Probar en imagen de ejemplo
if best_detection and best_classification:
    # Obtener imagen de prueba
    test_images = list((DETECTION_DATASET / 'test' / 'images').glob('*.jpg'))
    if len(test_images) == 0:
        test_images = list((DETECTION_DATASET / 'valid' / 'images').glob('*.jpg'))
    
    if len(test_images) > 0:
        test_img = random.choice(test_images)
        print(f"\n📸 Probando en: {test_img.name}")
        
        # Inferencia combinada
        results = combined_inference(
            str(test_img), best_detection, best_classification, class_names, device
        )
        
        # Mostrar resultados
        print(f"\n🎯 Detecciones ({len(results['detections'])})")
        for det in results['detections'][:5]:
            print(f"   - {det['class']}: {det['confidence']:.2%}")
        
        print(f"\n🏷️ Clasificaciones ({len(results['classifications'])})")
        for cls in results['classifications'][:10]:
            print(f"   - {cls['attribute']}: {cls['probability']:.2%}")
        
        # Visualizar
        img = Image.open(test_img)
        plt.figure(figsize=(10, 8))
        plt.imshow(img)
        plt.title(f"Imagen de prueba: {test_img.name}")
        plt.axis('off')
        plt.show()
    else:
        print("⚠️  No se encontraron imágenes de prueba")
else:
    print("⚠️  Modelos no cargados correctamente")

---
## 6️⃣ Exportar Modelos

In [ ]:
# Exportar modelos para producción
print("📤 Exportando modelos...")

# Exportar detección a ONNX
if best_detection:
    onnx_det_path = best_detection.export(format='onnx')
    print(f"   ✅ Detección exportada: {onnx_det_path}")

# Exportar clasificación a ONNX
if best_classification:
    onnx_cls_path = save_dir / 'model.onnx'
    dummy_input = torch.randn(1, 3, 224, 224).to(device)
    
    torch.onnx.export(
        best_classification,
        dummy_input,
        str(onnx_cls_path),
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
        opset_version=11
    )
    print(f"   ✅ Clasificación exportada: {onnx_cls_path}")

print("\n✅ Exportación completada")

---
## 📊 Resumen Final

In [ ]:
print("\n" + "="*60)
print("📊 RESUMEN DEL ENTRENAMIENTO")
print("="*60)

print("\n🎯 MODELO DE DETECCIÓN (YOLOv11)")
print(f"   Modelo: {DETECTION_CONFIG['model']}")
print(f"   Ubicación: {Path(DETECTION_CONFIG['project']) / DETECTION_CONFIG['name']}")

print("\n🏷️ MODELO DE CLASIFICACIÓN (Multi-label)")
print(f"   Backbone: {CLASSIFICATION_CONFIG['backbone']}")
print(f"   Mejor F1 Macro: {best_f1:.4f}")
print(f"   Ubicación: {save_dir}")

print("\n📁 Archivos generados:")
print(f"   - {Path(DETECTION_CONFIG['project']) / DETECTION_CONFIG['name'] / 'weights' / 'best.pt'}")
print(f"   - {save_dir / 'best.pt'}")

print("\n" + "="*60)
print("✅ ENTRENAMIENTO COMPLETO")
print("="*60)